# Integration Testing Strategies

## Overview

There are four main strategies for integration testing:

1. **Big-Bang**: Integrate all components at once
2. **Top-Down**: Integrate from top to bottom, using stubs
3. **Bottom-Up**: Integrate from bottom to top, using drivers
4. **Sandwich (Hybrid)**: Combine Top-Down and Bottom-Up

---

## 1. Big-Bang Integration

### Description
All modules are integrated simultaneously and tested as a whole.

### Pros
- Simple to implement
- No need for stubs or drivers

### Cons
- Difficult to locate defects
- All modules must be ready
- Testing can be overwhelming

### Example

In [ ]:
from src.service import Service

# Big-Bang: Test the complete system with all real components
def test_big_bang():
    service = Service()  # Uses real Storage and Notifier
    
    # Test the complete flow
    result = service.add_task("Big-Bang Task", "Testing complete integration")
    
    # Verify all components worked together
    tasks = service.get_tasks()
    notifications = service.get_notifications()
    
    assert result is True
    assert len(tasks) == 1
    assert len(notifications) == 1
    assert tasks[0]['title'] == "Big-Bang Task"
    assert "Big-Bang Task" in notifications[0]
    
    print("Big-Bang integration test passed!")

test_big_bang()

## 2. Top-Down Integration

### Description
Start testing from the top-level modules and work downwards. Lower-level modules are replaced with stubs.

### Pros
- Early testing of high-level logic
- Defects found in critical paths early
- Stubs are simple to create

### Cons
- Lower-level modules tested late
- Stubs may not represent real behavior accurately

### Example

In [ ]:
class StorageStub:
    """Stub for Storage in Top-Down testing."""
    def __init__(self):
        self.tasks = []
    def save(self, task):
        self.tasks.append(task)
        return True
    def get_all(self):
        return self.tasks.copy()

class NotifierStub:
    """Stub for Notifier in Top-Down testing."""
    def __init__(self):
        self.notifications = []
    def send(self, message):
        self.notifications.append(message)
        return True
    def get_notifications(self):
        return self.notifications.copy()

def test_top_down():
    # Top-Down: Test Service with stubs for Storage and Notifier
    storage_stub = StorageStub()
    notifier_stub = NotifierStub()
    service = Service(storage=storage_stub, notifier=notifier_stub)
    
    # Test the service logic
    result = service.add_task("Top-Down Task", "Testing with stubs")
    
    # Verify service interacted with stubs correctly
    assert result is True
    assert len(storage_stub.tasks) == 1
    assert len(notifier_stub.notifications) == 1
    
    print("Top-Down integration test passed!")

test_top_down()

## 3. Bottom-Up Integration

### Description
Start testing from the lowest-level modules and work upwards. Higher-level modules are replaced with drivers.

### Pros
- Early testing of utility modules
- Drivers can be more sophisticated than stubs
- Lower-level defects found early

### Cons
- High-level logic tested late
- Drivers can be complex

### Example

In [ ]:
from src.storage import Storage
from src.notifier import Notifier

def test_bottom_up_storage():
    """Bottom-Up: Test Storage module in isolation."""
    storage = Storage()
    
    # Test various scenarios
    valid_task = {'title': 'Valid Task', 'description': 'Test'}
    assert storage.save(valid_task) is True
    assert len(storage.get_all()) == 1
    
    # Test error cases
    try:
        storage.save({'title': '', 'description': 'Test'})
        assert False, "Should have raised ValueError"
    except ValueError:
        pass
    
    print("Bottom-Up Storage test passed!")

def test_bottom_up_notifier():
    """Bottom-Up: Test Notifier module in isolation."""
    notifier = Notifier()
    
    # Test various scenarios
    assert notifier.send("Test message") is True
    assert len(notifier.get_notifications()) == 1
    
    # Test error cases
    try:
        notifier.send("")
        assert False, "Should have raised ValueError"
    except ValueError:
        pass
    
    print("Bottom-Up Notifier test passed!")

test_bottom_up_storage()
test_bottom_up_notifier()

## 4. Sandwich (Hybrid) Integration

### Description
Combines Top-Down and Bottom-Up approaches. Test from both ends and meet in the middle.

### Pros
- Balances advantages of both approaches
- Parallel testing possible
- Reduces overall testing time

### Cons
- More complex to manage
- Requires both stubs and drivers

### Example

In [ ]:
def test_sandwich():
    """Sandwich: Combine real and stubbed components."""
    
    # Test with real Storage but stubbed Notifier
    real_storage = Storage()
    notifier_stub = NotifierStub()
    service = Service(storage=real_storage, notifier=notifier_stub)
    
    result = service.add_task("Sandwich Task 1", "Real storage, stub notifier")
    assert result is True
    assert len(real_storage.get_all()) == 1
    assert len(notifier_stub.notifications) == 1
    
    # Test with stubbed Storage but real Notifier
    storage_stub = StorageStub()
    real_notifier = Notifier()
    service = Service(storage=storage_stub, notifier=real_notifier)
    
    result = service.add_task("Sandwich Task 2", "Stub storage, real notifier")
    assert result is True
    assert len(storage_stub.tasks) == 1
    assert len(real_notifier.get_notifications()) == 1
    
    print("Sandwich integration test passed!")

test_sandwich()

## Strategy Comparison

| Strategy | When to Use | Complexity | Defect Localization |
|----------|-------------|------------|---------------------|
| Big-Bang | Small systems, simple integrations | Low | Poor |
| Top-Down | High-level logic critical, lower-level modules delayed | Medium | Good for high-level |
| Bottom-Up | Utility modules critical, high-level modules delayed | Medium | Good for low-level |
| Sandwich | Large systems, need parallel testing | High | Excellent |

---

## Recommendation

For most projects, **Sandwich integration** provides the best balance:
- Test critical low-level modules early (Bottom-Up)
- Test critical high-level logic early (Top-Down)
- Meet in the middle for complete integration testing